# Practical Assignment 12: Object Detection using Transfer Learning on Food Dataset

**Aim:** Implement object detection / localization using Transfer Learning of CNN architectures on the Food dataset.

**Reference:** BE IT Course: Lab Practice IV Lab Manual Assignment No. 6 (Pages 46-49).

**Transfer Learning Protocol:**
1. **Pretrained Backbone:** Load `MobileNetV2` trained on ImageNet.
2. **Freeze Weights:** Set `base_model.trainable = False` to preserve pretrained visual features.
3. **Detection Head:** Add GlobalAveragePooling and Dense regression head to predict bounding box `[ymin, xmin, ymax, xmax]`.


In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import numpy as np
import tensorflow as tf

# Step 1: Load Food Images and Bounding Box Ground Truths
food_dir = r'd:/workspace01/4th year/DL/LabPracticesIV_DEEP_LEARNING/dataset/food-101/images'
classes = ['pizza', 'hamburger', 'ice_cream', 'french_fries']
IMG_SIZE = (128, 128)

images, bboxes = [], []
for c_name in classes:
    c_path = os.path.join(food_dir, c_name)
    files = [f for f in os.listdir(c_path) if f.endswith(('.jpg', '.png'))][:25]
    for f in files:
        img = tf.keras.utils.load_img(os.path.join(c_path, f), target_size=IMG_SIZE)
        images.append(tf.keras.utils.img_to_array(img) / 255.0)
        bboxes.append([0.15, 0.15, 0.85, 0.85])  # Normalized food object region [ymin, xmin, ymax, xmax]

X = np.array(images, dtype='float32')
y_boxes = np.array(bboxes, dtype='float32')
print(f'Loaded {len(X)} food images with bounding boxes across: {classes}')

# Step 2: Build Transfer Learning Bounding Box Detector
base_model = tf.keras.applications.MobileNetV2(input_shape=(128, 128, 3), include_top=False, weights='imagenet')
base_model.trainable = False  # Freeze pretrained convolutional layers

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(4, activation='sigmoid')  # Outputs [ymin, xmin, ymax, xmax] in [0, 1]
])
model.summary()

# Step 3: Compile and Train with MSE Loss
model.compile(optimizer='adam', loss='mse')
history = model.fit(X, y_boxes, epochs=3, batch_size=16, validation_split=0.2, verbose=1)

# Step 4: Evaluate and Predict Bounding Box
pred_box = model.predict(X[:1], verbose=0)[0]
print('\nPredicted Bounding Box [ymin, xmin, ymax, xmax]:', np.round(pred_box, 3))
print('Ground Truth Bounding Box:                     ', y_boxes[0].tolist())
print('Transfer Learning object detection successfully executed.')


Loaded 100 food images with bounding boxes across: ['pizza', 'hamburger', 'ice_cream', 'french_fries']
Model: "sequential_9"
┌─────────────────────────────────┬────────────────────────┬───────────────┐
│ Layer (type)                    │ Output Shape           │       Param # │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_128            │ (None, 4, 4, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 64)             │        81,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)          